In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import date as dt

SILVER = "/Volumes/workspace/olist/silver"
GOLD   = "/Volumes/workspace/olist/gold"

# Lê o Silver
df_silver = spark.read.format("delta").load(f"{SILVER}/orders_enriched")

print(f"Registros lidos do Silver: {df_silver.count()}")
print(f"Colunas: {len(df_silver.columns)}")

In [0]:
# GOLD 1 — Faturamento por estado
df_faturamento_estado = df_silver \
    .filter(F.col("order_status") == "delivered") \
    .groupBy("customer_state") \
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(F.sum("price"), 2).alias("receita_produtos"),
        F.round(F.sum("freight_value"), 2).alias("receita_frete"),
        F.round(F.sum("price") + F.sum("freight_value"), 2).alias("receita_total"),
        F.round(F.avg("price"), 2).alias("ticket_medio")
    ) \
    .orderBy(F.col("receita_total").desc())

print(f"Estados únicos: {df_faturamento_estado.count()}")
display(df_faturamento_estado)

In [0]:
# GOLD 2 — Performance por categoria
df_categoria = df_silver \
    .filter(F.col("order_status") == "delivered") \
    .groupBy("product_category_name") \
    .agg(
        F.count("order_item_id").alias("total_itens_vendidos"),
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(F.sum("price"), 2).alias("receita_total"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.avg("freight_value"), 2).alias("frete_medio")
    ) \
    .orderBy(F.col("receita_total").desc())

print(f"Categorias únicas: {df_categoria.count()}")
display(df_categoria.limit(15))

In [0]:
# GOLD 3 — Análise de entregas por estado
df_entregas = df_silver \
    .filter(F.col("order_status") == "delivered") \
    .filter(F.col("order_delivered_customer_date").isNotNull()) \
    .filter(F.col("order_purchase_timestamp").isNotNull()) \
    .withColumn("prazo_entrega_dias",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    ) \
    .groupBy("customer_state") \
    .agg(
        F.count("order_id").alias("total_entregas"),
        F.round(F.avg("prazo_entrega_dias"), 1).alias("prazo_medio_dias"),
        F.min("prazo_entrega_dias").alias("prazo_minimo_dias"),
        F.max("prazo_entrega_dias").alias("prazo_maximo_dias")
    ) \
    .orderBy(F.col("prazo_medio_dias").asc())

print(f"Estados: {df_entregas.count()}")
display(df_entregas)

In [0]:
# GOLD 4 — Top clientes por LTV corrigido
df_top_clientes = df_silver \
    .filter(F.col("order_status") == "delivered") \
    .groupBy("customer_unique_id") \
    .agg(
        F.first("customer_state").alias("customer_state"),
        F.first("customer_city").alias("customer_city"),
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(F.sum("price"), 2).alias("receita_total"),
        F.round(F.avg("price"), 2).alias("ticket_medio"),
        F.round(F.sum(F.col("price") + F.col("freight_value")), 2).alias("ltv_total")
    ) \
    .orderBy(F.col("ltv_total").desc())

print(f"Clientes únicos: {df_top_clientes.count()}")
display(df_top_clientes.limit(10))

In [0]:
from datetime import date as dt

tabelas_gold = [
    (df_faturamento_estado, "faturamento_por_estado"),
    (df_categoria,          "performance_por_categoria"),
    (df_entregas,           "analise_entregas"),
    (df_top_clientes,       "top_clientes_ltv")
]

for df, nome in tabelas_gold:
    df.withColumn("_processing_date", F.lit(str(dt.today()))) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{GOLD}/{nome}")
    
    total = spark.read.format("delta").load(f"{GOLD}/{nome}").count()
    print(f"{nome}: {total} registros gravados")

print("\nSprint 5 — Camada Gold criada com sucesso!")